In [1]:
#Importing the necessary libraries
import jsonlines
import pandas as pd
import numpy as np
import re
from keras.layers import Dense, Dropout
from keras.models import Sequential
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split

In [2]:
#Importing the data
newlist = []
with jsonlines.open('categorized-comments.jsonl') as f:
    for obj in f.iter(type = dict, skip_invalid = True):
        newlist.append(obj)

comments = pd.DataFrame(newlist)
del newlist

In [3]:
#Balancing the classes by taking 15% of each category
#The 15% will be randomly sampled to reduce the size of the dataset
dflist = []
for cat in np.unique(comments['cat']):
    dflist.append(comments[comments['cat'] == cat].sample(frac = 0.15))
comments = pd.concat(dflist)
del dflist

In [4]:
#Cleaning the text by removing punctuation and stopwords and stemming

stemmer = SnowballStemmer('english')
words = stopwords.words('english')

comments['cleaned'] = comments['txt'].apply(lambda x: ' '.join([stemmer.stem(i) for i in re.sub("[^a-zA-Z]", " ", x).split() if i not in words]).lower())
del comments['txt']

In [5]:
#Generating a TFIDF matrix from the cleaned text as input to the keras neural network model

count = TfidfVectorizer()
X = count.fit_transform(comments['cleaned']).toarray()

In [6]:
#Splitting the dataset into training and testing sets 
#40% of the data is set aside for testing due to memory limits on training array size

X_train, X_test, y_train, y_test = train_test_split(X, comments['cat'], test_size = 0.4)
del X

In [7]:
#Turning the categorical target vectors into numerical vectors

y_train = y_train.astype('category').cat.codes
y_test = y_test.astype('category').cat.codes

In [8]:
#Defining a function to build, train, and test the model with 4 layers

def model_build_train_test(X_train, y_train, X_test, batch_size, epochs):    
    #Defining the model and adding the 3 layers, with neuron numbers and activation functions specified
    model = Sequential()
    model.add(Dense(720, activation = 'relu', input_dim = X_train.shape[1]))
    model.add(Dropout(0.3))
    model.add(Dense(120, activation = 'relu'))
    model.add(Dropout(0.3))
    model.add(Dense(20, activation = 'relu'))
    model.add(Dropout(0.3))
    model.add(Dense(3, activation = 'softmax'))
    
    #Compiling the model and displaying the training accuracy of each epoch
    model.compile(
              loss = 'sparse_categorical_crossentropy',
              optimizer = 'adam',
              metrics = ['accuracy'])
    
    #Fitting the model to the training data
    model.fit(X_train, y_train, batch_size = batch_size, epochs = epochs)
    
    #Testing the model on the testing set and returning the predictions
    y_pred = np.argmax(model.predict(X_test), axis = 1)
    return y_pred

In [9]:
#Running the model function
y_pred = model_build_train_test(X_train, y_train, X_test, 5000, 5)

Epoch 1/5
11/11 [==============================] - 76s 5s/step - loss: 1.0359 - accuracy: 0.5960
Epoch 2/5
11/11 [==============================] - 45s 4s/step - loss: 0.7411 - accuracy: 0.7159
Epoch 3/5
11/11 [==============================] - 42s 4s/step - loss: 0.6620 - accuracy: 0.7213
Epoch 4/5
11/11 [==============================] - 41s 4s/step - loss: 0.5644 - accuracy: 0.7663
Epoch 5/5
11/11 [==============================] - 38s 3s/step - loss: 0.4627 - accuracy: 0.8299


In [12]:
#Displaying the result metrics by comparing the predicted categories to the true categories
#The training accuracies were checked as well, and it was confirmed that overfitting was not occurring

print('Scores')
print('------')
print('Accuracy: {} percent'.format(round(accuracy_score(y_test, y_pred), 3)*100))
print('Precision: {} percent'.format(round(precision_score(y_test, y_pred, average = 'weighted'), 3)*100))
print('Recall: {} percent'.format(round(recall_score(y_test, y_pred, average = 'weighted'), 3)*100))
print('F1 Score: {} percent \n'.format(round(f1_score(y_test, y_pred, average = 'weighted'), 3)*100))

print('Confusion matrix \n---------------\n')
print(confusion_matrix(y_test, y_pred))

Scores
------
Accuracy: 82.5 percent
Precision: 78.9 percent
Recall: 82.5 percent
F1 Score: 80.0 percent 

Confusion matrix 
---------------

[[    0   152  1349]
 [    0  5264  3679]
 [    0  1187 24758]]
